# XGBoost+SHAP

In [ ]:
import pandas as pd
import os
import rasterio
import numpy as np
import pandas as pd
from scipy.stats import zscore, t


file_path = r"H:\Spatial_attribution_analysis\data.xlsx"
df = pd.read_excel(file_path)

df = df.dropna()

y = df.iloc[:, 1]  # Read first column
x = df.iloc[:, 2:]  # Read all remaining columns


# Output shape check
print("y shape:", y.shape)
print("x shape:", x.shape)

In [ ]:
import shap
import pandas as pd
from scipy.stats import zscore
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Split dataset into training and test sets (X is feature data, y is target variable)
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# Define XGBoost regression model
model = XGBRegressor(objective='reg:squarederror', random_state=42, tree_method='exact')

# Define parameter distribution
param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 9],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.8, 1.0]
}

# Perform hyperparameter optimization using RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=50,  # 50 search iterations
    cv=5,  # 5-fold cross-validation
    scoring='r2',  # Evaluation metric: R²
    n_jobs=-1,  # Use all available CPUs for parallel computation
    random_state=42
)

# Run hyperparameter search
random_search.fit(X_train, y_train)

# Output best parameters and best R² score
best_params = random_search.best_params_
best_score = random_search.best_score_
print(f"Best parameters: {best_params}")
print(f"Best cross-validation R² score: {best_score}")

# Train final model with best parameters
best_model = XGBRegressor(objective='reg:squarederror', random_state=42, tree_method='exact', **best_params)
best_model.fit(X_train, y_train)

# Calculate R² for training and test sets
r2_train = r2_score(y_train, best_model.predict(X_train))
r2_test = r2_score(y_test, best_model.predict(X_test))

print(f"Training set R²: {r2_train}, Test set R²: {r2_test}")

# For entire dataset
explainer = shap.TreeExplainer(best_model)

var_list = x.columns.tolist()

shap_values = explainer(x)

# Save SHAP values
df_shap = pd.DataFrame(shap_values.values, columns=x.columns)
print(df_shap.head())

# Save results
df_shap.to_csv(r"H:\Spatial_attribution_analysis\shap_values.csv", index=False)
mean_shap_values = np.abs(shap_values.values).mean(axis=0)

df_shap_mean = pd.DataFrame({
    "Feature": var_list,
    "Mean SHAP Value": mean_shap_values
}).sort_values(by="Mean SHAP Value", ascending=False)

mse = mean_squared_error(y, best_model.predict(x))
r2 = best_model.score(x, y)
print(f"MSE: {mse}, R²: {r2}")

# Plotting
shap.initjs()
shap.summary_plot(shap_values, x, cmap="plasma")
shap.plots.beeswarm(shap_values, max_display=21)  # Bee swarm plot for SHAP values
shap.plots.heatmap(shap_values, max_display=21)
shap.plots.bar(shap_values, max_display=21)  # Bar plot for feature importance

shap.plots.scatter(shap_values[:, "WUE"], show=False)  # Scatter plot for a specific feature (e.g., "bio")
shap.plots.scatter(shap_values[:, "ALAN"], show=False)
shap.plots.scatter(shap_values[:, "HDI"], show=False)